# Intraday patterns

Behaviour by hour, weekday and trading session, on the configured broker clock.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.research` and read back from
`results/statistical_research/<timeframe>/`. Generate those first:

```
xq research-summary --timeframe 5m
xq research-summary --all
```

Nothing here is a trading signal. No transaction costs are applied anywhere.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.research import load_research_config

config = load_config(ROOT / "config" / "data.yaml")
research = load_research_config(ROOT / "config" / "research.yaml")

TIMEFRAME = "5m"          # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"time basis : {research.intraday.time_basis}")
print(f"results    : {research.results_path}")

In [ ]:
import json

results = research.results_dir(TIMEFRAME)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME}. Run:  xq research-summary --timeframe {TIMEFRAME}"
    )

def table(name):
    """Load a previously-generated research table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
print(f"generated : {summary['provenance']['generated_utc']}")
print(f"span      : {summary['bars']['first_timestamp']} -> {summary['bars']['last_timestamp']}")
print(f"bars      : {summary['bars']['count']:,}")

## Which clock?

Every grouping below uses the configured time basis. It is **not** UTC unless `intraday.time_basis` says so.

In [ ]:
print("time basis :", summary["provenance"]["timezone"]["time_basis"])
print("timezone   :", summary["provenance"]["timezone"]["description"])

## By hour of day

In [ ]:
hourly = table("intraday")
hourly.select([c for c in ("hour", "bars", "mean_return", "return_std",
                           "mean_abs_return", "mean_spread", "mean_tick_count",
                           "extreme_return_frequency") if c in hourly.columns])

## By weekday

Weekend bars are reported rather than dropped: XAUUSD is normally closed then, so their presence is itself information.

In [ ]:
weekday = table("weekday")
weekday.select([c for c in ("weekday", "bars", "mean_return", "return_std",
                            "mean_abs_return", "mean_spread", "mean_tick_count")
                if c in weekday.columns])

## By session

Boundaries are configuration, not official hours, and are **not** daylight-saving adjusted.

In [ ]:
session = table("session")
session.select([c for c in ("session", "bars", "mean_return", "return_std",
                            "mean_abs_return", "mean_spread", "mean_tick_count")
                if c in session.columns])

## Spread by hour

This matters for everything downstream: an effect smaller than the spread is not capturable.

In [ ]:
table('spread_by_hour')

## Plot

In [ ]:
from IPython.display import Image, display

path = research.plots_dir(TIMEFRAME) / "intraday_profile.png"
if path.exists():
    display(Image(str(path)))